# Auskunft Layer - Server

In [1]:
import logging


class PhoneDB:
    def __init__(self):
        self._logger = logging.getLogger("vs2lab.lab1.Telefonauskunft_Server.PhoneDB")
        self._phones = {"Scooby Doo": "555-1234", "Shaggy": "555-5678"}
        self._logger.info("phone database initialized.")
    def process_request(self, request:str, **kwargs)->str:
        if request == "GETALL":
            return self._GETALL()
        elif request == "GET":
            return self._GET(kwargs.get("name", ""))
        else:
            self._logger.error(f"Unknown request: {request}")
            return "ERROR: Unknown request"
        
    def _GETALL(self)->str:
        return str(self._phones)
    
    def _GET(self, name:str)->str:
        if not name:
            self._logger.error("No name provided for GET request.")
            return "ERROR: No name provided"
        for key in self._phones.keys():
            if key == name:
                return str({key: self._phones[key]})
            
        self._logger.warning(f"Name not found: {name}")
        return "ERROR: Name not found"

# Transport Layer - Server

In [2]:
import logging
import clientserver
import socket

class Transport:
    def __init__(self):
        self._logger = logging.getLogger("vs2lab.lab1.Telefonauskunft_Server.Transport")
        self.phone_db = PhoneDB()
        self._serving = True
        self.sock = clientserver.Server().sock
        self._serve(True)

    def _serve(self, debug=False):
        """Serve echo"""
        self.sock.listen(1)
        while self._serving:
            self._logger.info("Waiting for connection...")
            try:
                connection, address = self.sock.accept()
                self._logger.info(f"Connection established with {address}.")
                while True:
                    data = connection.recv(1024)
                    if not data:
                        self._logger.info("No data received, closing connection.")
                        break
                    decoded = data.decode("ascii")
                    parts = decoded.split("::")
                    result = self.phone_db.process_request(parts[0], name=parts[1])
                    connection.send(result.encode("ascii"))
                connection.close()
                self._logger.info("Connection closed.")
            except socket.timeout:
                pass
        self.sock.close()
        self._logger.info("Server down.")


# Beispiel Nutzung - Server

In [3]:
server = Transport()

2025-10-08 09:30:05,713 - vs2lab.lab1.Telefonauskunft_Server.PhoneDB - INFO - phone database initialized.
2025-10-08 09:30:05,715 - vs2lab.lab1.clientserver.Server - INFO - Server bound to socket <socket.socket fd=58, family=2, type=1, proto=0, laddr=('127.0.0.1', 50007)>
2025-10-08 09:30:05,716 - vs2lab.lab1.Telefonauskunft_Server.Transport - INFO - Waiting for connection...
2025-10-08 09:30:08,719 - vs2lab.lab1.Telefonauskunft_Server.Transport - INFO - Waiting for connection...
2025-10-08 09:30:10,702 - vs2lab.lab1.Telefonauskunft_Server.Transport - INFO - Connection established with ('127.0.0.1', 43236).
2025-10-08 09:30:10,704 - vs2lab.lab1.Telefonauskunft_Server.Transport - INFO - No data received, closing connection.
2025-10-08 09:30:10,705 - vs2lab.lab1.Telefonauskunft_Server.Transport - INFO - Connection closed.
2025-10-08 09:30:10,705 - vs2lab.lab1.Telefonauskunft_Server.Transport - INFO - Waiting for connection...
2025-10-08 09:30:10,706 - vs2lab.lab1.Telefonauskunft_Server.T

KeyboardInterrupt: 

# Unit Tests - Server

In [4]:
import unittest

# Tests for PhoneDB defined in this notebook
class TestPhoneDB(unittest.TestCase):
    def setUp(self):
        self.db = PhoneDB()

    def test_getall_contains_scooby(self):
        result = self.db.process_request('GETALL')
        # GETALL returns a string representation of the dict; check expected substring
        self.assertIn('Scooby Doo', result)
        self.assertIn('555-1234', result)

    def test_get_existing(self):
        result = self.db.process_request('GET', name='Scooby Doo')
        self.assertIn('Scooby Doo', result)
        self.assertIn('555-1234', result)

    def test_get_no_name(self):
        result = self.db.process_request('GET', name='')
        self.assertEqual(result, 'ERROR: No name provided')

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

2025-10-08 09:30:48,206 - vs2lab.lab1.Telefonauskunft_Server.PhoneDB - INFO - phone database initialized.
.2025-10-08 09:30:48,207 - vs2lab.lab1.Telefonauskunft_Server.PhoneDB - INFO - phone database initialized.
2025-10-08 09:30:48,208 - vs2lab.lab1.Telefonauskunft_Server.PhoneDB - ERROR - No name provided for GET request.
.2025-10-08 09:30:48,209 - vs2lab.lab1.Telefonauskunft_Server.PhoneDB - INFO - phone database initialized.
.
----------------------------------------------------------------------
Ran 3 tests in 0.004s

OK
